# Week 6: Probabilistic and Hierarchical Forecasting
## In-Class Exercises

**Objective.** Produce forecasts that carry honest uncertainty, check that the uncertainty is honest, and make a set of forecasts add up across a hierarchy.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | Prediction intervals from a fitted model, and the coverage check that deflates them. |
| 2 | Blanks we fill in together | Three ways to build an 80% interval, scored against each other. |
| 3 | On your own, ~15 min | Quantile regression and pinball loss. |

In [1]:
!pip install -q statsforecast lightgbm pandas plotly scikit-learn mlforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.8 MB/s eta 0:00:00


In [2]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive, SimpleExponentialSmoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"   # try "plotly_dark", "ggplot2", "simple_white"

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
air = (pd.read_csv(URL, parse_dates=["Month"])
         .rename(columns={"Month": "ds", "Passengers": "y"})
         .assign(unique_id="airline"))

h, m = 24, 12
train, test = air.iloc[:-h], air.iloc[-h:]


def band(fig, x, lo, hi, name, color="rgba(31,119,180,0.20)"):
    """Add a shaded interval to a figure. Two traces: invisible floor, filled ceiling."""
    fig.add_scatter(x=x, y=lo, mode="lines", line_width=0, showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=x, y=hi, mode="lines", line_width=0, fill="tonexty",
                    fillcolor=color, name=name)
    return fig


train.tail(2)

,ds,y,unique_id
118,1958-11-01,310,airline
119,1958-12-01,337,airline


---
## Part 1. An interval, and whether to believe it

A point forecast summarizes a distribution, and it is rarely the part of the distribution a decision needs. Inventory and capacity planning need a high quantile. A revenue forecast needs the mean *and* a downside.

`statsforecast` gives intervals with a `level` argument. Watch what happens when I check them.

In [3]:
sf = StatsForecast(models=[AutoETS(season_length=m)], freq="MS")
fc = sf.forecast(df=train, h=h, level=[80, 95])
fc.head(3)

,unique_id,ds,AutoETS,AutoETS-lo-95,AutoETS-lo-80,AutoETS-hi-80,AutoETS-hi-95
0,airline,1959-01-01,348.191040,312.120819,324.605988,371.776093,384.261261
1,airline,1959-02-01,346.660034,308.778656,321.890747,371.429352,384.541412
2,airline,1959-03-01,395.853210,355.732178,369.619476,422.086945,435.974243


In [4]:
ev = test.merge(fc, on=["unique_id", "ds"])

fig = go.Figure()
band(fig, ev["ds"], ev["AutoETS-lo-95"], ev["AutoETS-hi-95"], "95%", "rgba(31,119,180,0.15)")
band(fig, ev["ds"], ev["AutoETS-lo-80"], ev["AutoETS-hi-80"], "80%", "rgba(31,119,180,0.35)")
fig.add_scatter(x=train["ds"].tail(48), y=train["y"].tail(48), name="train",
                line_color="black")
fig.add_scatter(x=ev["ds"], y=ev["y"], name="actual", line_color="black", line_dash="dash")
fig.add_scatter(x=ev["ds"], y=ev["AutoETS"], name="forecast")
fig.update_layout(title="AutoETS forecast with 80% and 95% intervals")
fig.show()

for lv in [80, 95]:
    inside = ((ev["y"] >= ev[f"AutoETS-lo-{lv}"]) & (ev["y"] <= ev[f"AutoETS-hi-{lv}"])).mean()
    width = (ev[f"AutoETS-hi-{lv}"] - ev[f"AutoETS-lo-{lv}"]).mean()
    print(f"nominal {lv}%  ->  empirical coverage {inside:.0%}, mean width {width:.1f}")

nominal 80%  ->  empirical coverage 21%, mean width 73.4
nominal 95%  ->  empirical coverage 33%, mean width 112.3


In [5]:
sf = StatsForecast(
    models=[
        AutoETS(season_length=m, model="MAM", damped=False, alias="AutoETS"),
        ],
    freq="MS",
)
fc = sf.forecast(df=train, h=h, level=[80, 95])
fc.head(3)

ev = test.merge(fc, on=["unique_id", "ds"])

fig = go.Figure()
band(fig, ev["ds"], ev["AutoETS-lo-95"], ev["AutoETS-hi-95"], "95%", "rgba(31,119,180,0.15)")
band(fig, ev["ds"], ev["AutoETS-lo-80"], ev["AutoETS-hi-80"], "80%", "rgba(31,119,180,0.35)")
fig.add_scatter(x=train["ds"].tail(48), y=train["y"].tail(48), name="train",
                line_color="black")
fig.add_scatter(x=ev["ds"], y=ev["y"], name="actual", line_color="black", line_dash="dash")
fig.add_scatter(x=ev["ds"], y=ev["AutoETS"], name="forecast")
fig.update_layout(title="AutoETS forecast with 80% and 95% intervals")
fig.show()

for lv in [80, 95]:
    inside = ((ev["y"] >= ev[f"AutoETS-lo-{lv}"]) & (ev["y"] <= ev[f"AutoETS-hi-{lv}"])).mean()
    width = (ev[f"AutoETS-hi-{lv}"] - ev[f"AutoETS-lo-{lv}"]).mean()
    print(f"nominal {lv}%  ->  empirical coverage {inside:.0%}, mean width {width:.1f}")

nominal 80%  ->  empirical coverage 83%, mean width 61.5
nominal 95%  ->  empirical coverage 96%, mean width 94.1


**Notice:**

Coverage on 24 test points is a noisy estimate, so do not over-read one number. The direction of the miss is consistent across real datasets though: **published intervals are too narrow.** Three standard reasons:

1. The interval uses the model's estimate of $\sigma$ and ignores uncertainty in the *parameters*.
2. It ignores uncertainty about the *model*. You chose ETS after looking at the data; that choice had a variance too, and it appears nowhere in the arithmetic.
3. It assumes the future error distribution matches the past one. Structural breaks say otherwise.

The fan widening with horizon is right and important. A 12-month forecast with a constant-width band was built wrong.

---
## Part 2. Three ways to make an 80% interval

Same point forecast, three theories of uncertainty.

- **Normal**: $\hat{y}_{t+k} \pm z_{0.9} \cdot \hat\sigma_k$, with $\hat\sigma$ from in-sample residuals scaled by $\sqrt{k}$. Cheap, symmetric, Gaussian.
- **Bootstrap**: resample past residuals, roll them forward through the forecast path many times, take empirical quantiles. Keeps the residuals' shape and skew.
- **Split conformal**: hold out a calibration window, measure absolute errors *at each horizon*, use the 80th percentile as the half-width. Distribution-free, with a finite-sample guarantee under exchangeability.

Fill in the `TODO`s with me.

In [6]:
# Point forecasts and in-sample residuals from the same model.
sf_fit = StatsForecast(    models=[
        AutoETS(season_length=m, model="MAM", damped=False, alias="AutoETS"),
        ], freq="MS")
res = sf_fit.forecast(df=train, h=h, fitted=True)
fitted = sf_fit.forecast_fitted_values()

point = res["AutoETS"].to_numpy()
resid = (fitted["y"] - fitted["AutoETS"]).dropna().to_numpy()
actual = test["y"].to_numpy()
LEVEL = 80
alpha = 1 - LEVEL / 100

print("residual sd:", round(resid.std(), 2), " skew:", round(pd.Series(resid).skew(), 2))

residual sd: 14.08  skew: -0.33


In [7]:
from scipy.stats import norm

# --- 1. Normal ---
z = norm.ppf(1 - alpha / 2)
sigma_k = resid.std() * np.sqrt(np.arange(1, h + 1))
normal_lo, normal_hi = point - z * sigma_k, point + z * sigma_k

# --- 2. Bootstrap ---
rng = np.random.default_rng(0)
B = 2000
# TODO - draw B paths. Each path is the point forecast plus a cumulative sum of
# residuals sampled with replacement (the cumsum is what makes the fan widen).
#   Hint: rng.choice(resid, size=(B, h), replace=True).cumsum(axis=1)
paths = point + rng.choice(resid, size=(B, h), replace=True).cumsum(axis=1)
boot_lo = np.percentile(paths, 100 * alpha / 2, axis=0)
boot_hi = np.percentile(paths, 100 * (1 - alpha / 2), axis=0)

# --- 3. Split conformal ---
# Calibration: refit on data up to a cut point, forecast h steps, record |error| per horizon.
cal_errors = []
for cut in range(len(train) - 4 * h, len(train) - h, 3):
    sub = train.iloc[:cut]
    f = StatsForecast(models=[AutoETS(season_length=m)], freq="MS").forecast(df=sub, h=h)
    truth = train.iloc[cut:cut + h]["y"].to_numpy()
    cal_errors.append(np.abs(truth - f["AutoETS"].to_numpy()))
cal_errors = np.vstack(cal_errors)

# TODO - the conformal half-width at horizon k is the (1 - alpha) quantile of the
# absolute calibration errors at that horizon.
q_k = np.quantile(cal_errors, 1 - alpha, axis=0)


conf_lo, conf_hi = point - q_k, point + q_k

<details>
<summary><b>Show the two filled-in lines</b></summary>

```python
paths = point + rng.choice(resid, size=(B, h), replace=True).cumsum(axis=1)

q_k = np.quantile(cal_errors, 1 - alpha, axis=0)
```
</details>

In [8]:
def report(lo, hi, label):
    inside = ((actual >= lo) & (actual <= hi)).mean()
    return {"method": label, "coverage": f"{inside:.0%}", "mean width": round(np.mean(hi - lo), 1)}

print(pd.DataFrame([report(normal_lo, normal_hi, "normal"),
                    report(boot_lo, boot_hi, "bootstrap"),
                    report(conf_lo, conf_hi, "split conformal")]).to_string(index=False))

for lo, hi, name in [(normal_lo, normal_hi, "normal"),
                     (boot_lo, boot_hi, "bootstrap"),
                     (conf_lo, conf_hi, "conformal")]:
    fig = go.Figure()
    band(fig, test["ds"], lo, hi, f"{LEVEL}% band")
    fig.add_scatter(x=test["ds"], y=actual, name="actual", line_color="black", line_dash="dash")
    fig.add_scatter(x=test["ds"], y=point, name="point forecast")
    fig.update_layout(title=f"{name} {LEVEL}% interval", height=320)
    fig.show()

         method coverage  mean width
         normal      88%  121.200000
      bootstrap      58%  117.900002
split conformal      83%   92.000000


**Reading the table.**

- Coverage alone never settles the argument. An interval from minus infinity to plus infinity has perfect coverage and zero value. Report **coverage and width** as a pair.
- Bootstrap intervals inherit the residuals' asymmetry. If your errors skew, the interval should too, and the normal one cannot.
- Conformal's guarantee is real but conditional on **exchangeability**, which time series violate as soon as there is a trend or regime change. Calibration windows sitting just before the test period are the best available defense, not a proof.
- That calibration loop cost a dozen model fits. Uncertainty quantification is not free, which is the real reason it gets skipped.

---
## Part 3. Quantiles directly, and the loss that scores them

About 15 minutes.

Rather than deriving an interval from a mean forecast, model the quantiles directly. LightGBM does this with `objective="quantile", alpha=tau`.

The scoring rule is **pinball loss**. For quantile $\tau$, forecast $q$, actual $y$:

$$L_\tau(y, q) = \begin{cases}\tau\,(y - q) & y \ge q \\ (1 - \tau)\,(q - y) & y < q\end{cases}$$

It penalizes misses on the expensive side harder, and it is *minimized in expectation by the true $\tau$-quantile*. That property makes it the right scorer rather than an arbitrary one.

**Tasks.**

1. Finish `pinball(y, q, tau)`.
2. Build the lag feature matrix (lags 1, 12, 13) and fit LightGBM at $\tau \in \{0.1, 0.5, 0.9\}$.
3. Forecast the test window and plot all three quantile paths against the actuals.
4. Compute pinball loss at each $\tau$ for your model and for the AutoETS quantiles from Part 1, and say who wins where.
5. Check for crossing. If $q_{0.1} > q_{0.9}$ anywhere, explain the cause and the fix.

In [9]:
from lightgbm import LGBMRegressor

def pinball(y, q, tau):
    """Mean pinball loss at quantile tau."""
    y, q = np.asarray(y, float), np.asarray(q, float)
    # TODO
    return ...


LAGS = [1, 12]

def lag_matrix(series, lags=LAGS):
    s = pd.Series(np.asarray(series, float))
    X = pd.DataFrame({f"lag{L}": s.shift(L) for L in lags})
    keep = X.notna().all(axis=1)
    return X[keep], s[keep]

In [13]:
# YOUR CODE HERE - tasks 2 through 5
def pinball(y, q, tau):
    y, q = np.asarray(y, float), np.asarray(q, float)
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))


import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from scipy.stats import norm

# mlforecast expects long format: unique_id, ds, y
df_tr = train[["ds", "y"]].assign(unique_id="series_1")
freq = pd.infer_freq(df_tr["ds"])          # or set explicitly, e.g. "MS", "h", "D"

taus = [0.1, 0.5, 0.9]
models = {
    f"q{int(t*100)}": LGBMRegressor(
        objective="quantile", alpha=t,
        n_estimators=200, learning_rate=0.05,
        num_leaves=8, min_child_samples=5,   # small data: let the trees split on month
        verbosity=-1,
    )
    for t in taus
}

mlf = MLForecast(
    models=models,             # same quantile LGBMs as before
    freq="MS",
    lags=[12],                 # same month last year
    date_features=["month"],
)

mlf.fit(df_tr)
fc = mlf.predict(h)

Q = np.sort(fc[[f"q{int(t*100)}" for t in taus]].to_numpy().T, axis=0)
qfc = {t: Q[i] for i, t in enumerate(taus)}

# collect quantiles and enforce monotonicity (removes crossings)
Q = np.sort(fc[[f"q{int(t*100)}" for t in taus]].to_numpy().T, axis=0)
qfc = {t: Q[i] for i, t in enumerate(taus)}

fig = go.Figure()
fig.add_scatter(x=test["ds"], y=actual, name="actual", line_color="black", line_dash="dash")
for tau in taus:
    fig.add_scatter(x=test["ds"], y=qfc[tau], name=f"tau={tau}")
fig.update_layout(title="Quantile forecasts")
fig.show()

for tau in taus:
    ets_q = point + norm.ppf(tau) * sigma_k
    print(f"tau={tau}  LGBM {pinball(actual, qfc[tau], tau):.2f}   AutoETS {pinball(actual, ets_q, tau):.2f}")

crossed = (qfc[0.1] > qfc[0.9]).sum()
print("crossings:", crossed)



tau=0.1  LGBM 8.08   AutoETS 6.57
tau=0.5  LGBM 26.44   AutoETS 8.08
tau=0.9  LGBM 26.61   AutoETS 6.48
crossings: 0


<details>
<summary><b>Solution</b></summary>

```python
def pinball(y, q, tau):
    y, q = np.asarray(y, float), np.asarray(q, float)
    d = y - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))


ytr = train["y"].to_numpy()
X, tgt = lag_matrix(ytr)

taus = [0.1, 0.5, 0.9]
qfc = {}
for tau in taus:
    mdl = LGBMRegressor(objective="quantile", alpha=tau, n_estimators=300, verbosity=-1).fit(X, tgt)
    hist = list(ytr)
    preds = []
    for _ in range(h):
        row = [[hist[-L] for L in LAGS]]
        nxt = float(mdl.predict(row)[0])
        preds.append(nxt)
        hist.append(nxt)          # note: recursion uses the tau path, not the median path
    qfc[tau] = np.array(preds)

fig = go.Figure()
fig.add_scatter(x=test["ds"], y=actual, name="actual", line_color="black", line_dash="dash")
for tau in taus:
    fig.add_scatter(x=test["ds"], y=qfc[tau], name=f"tau={tau}")
fig.update_layout(title="Quantile forecasts")
fig.show()

for tau in taus:
    ets_q = point + norm.ppf(tau) * sigma_k
    print(f"tau={tau}  LGBM {pinball(actual, qfc[tau], tau):.2f}   AutoETS {pinball(actual, ets_q, tau):.2f}")

crossed = (qfc[0.1] > qfc[0.9]).sum()
print("crossings:", crossed)
```

**On crossing.** The three models are fitted independently, so nothing enforces monotonicity. The cheap fix is sorting the quantiles at each timestamp (isotonic rearrangement), which is provably no worse under pinball loss. The principled fix is one model with a multi-quantile loss, which is what `MQLoss` did in Week 5.

**On recursion.** Feeding the $\tau = 0.9$ prediction back in compounds an *upper quantile* rather than a central path, so the 0.9 line drifts optimistically. Commonly missed. Recursive quantile forecasting needs simulation, not point recursion.
</details>

---
## Wrap-up

1. **Coverage and width, always together.** Either alone is gameable.
2. **Bootstrap keeps shape; conformal drops assumptions; normal is fast.** Pick knowingly.
3. **Pinball and CRPS are proper scoring rules.** They are how you compare distributions rather than points.
4. **Reconciliation is a projection.** MinT beats bottom-up because it uses information from every level.
5. **The quantile you report should come from the cost of being wrong**, not from convention.